# Deep Learning LSTM and Transformer

input : DS-D dataset train.csv and test.csv

### 1. Import Library

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

### 2. Data Loading & Windowing Strategy

In [4]:
# Configuration
PROCESSED_DIR = Path("../data/experiments/DS-D")
WINDOW_SIZE = 30
BATCH_SIZE = 64
EPOCHS = 50
FEATURES = ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 
            's_14', 's_15', 's_17', 's_20', 's_21', 'health_index'] 
# Note: Add your rolling mean/std columns to FEATURES if you want to include them.

def create_windows(data, window_size, feature_cols, target_col):
    X, y = [], []
    # Assuming 'unit_nr' or similar exists to group engines. 
    # If not present in your CSV, we treat the whole file as one sequence (less ideal).
    for i in range(len(data) - window_size):
        X.append(data[feature_cols].iloc[i:i+window_size].values)
        y.append(data[target_col].iloc[i+window_size])
    return np.array(X), np.array(y)

train_df = pd.read_csv(PROCESSED_DIR/"train.csv")
test_df = pd.read_csv(PROCESSED_DIR/"test.csv")


X_train_raw, y_train_raw = create_windows(train_df, WINDOW_SIZE, FEATURES, 'RUL_clipped')
X_test_raw, y_test_raw = create_windows(test_df, WINDOW_SIZE, FEATURES, 'RUL_clipped')

# Convert to Tensors
X_train = torch.tensor(X_train_raw, dtype=torch.float32)
y_train = torch.tensor(y_train_raw, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(X_test_raw, dtype=torch.float32)
y_test = torch.tensor(y_test_raw, dtype=torch.float32).view(-1, 1)

### 3. The LSTM Model Architecture

In [5]:
class RULPredictorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(RULPredictorLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        # We take the output of the last time step
        out = self.fc(out[:, -1, :])
        return out

model = RULPredictorLSTM(input_dim=len(FEATURES), hidden_dim=64, num_layers=2)

### 4. The Transformer Model Architecture

In [6]:
class RULTransformer(nn.Module):
    def __init__(self, input_dim, d_model=96, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.embed = nn.Linear(input_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=192, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        z = self.embed(x)
        z = self.encoder(z)
        z = z.mean(dim=1) # Global average pooling
        return self.head(z)

### 5. Training & Evaluation Logic

In [7]:
def train_and_evaluate(model, X_tr, y_tr, X_te, y_te, name="Model"):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    
    print(f"\n--- Training {name} ---")
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_tr)
        loss = torch.sqrt(criterion(outputs, y_tr)) # RMSE
        loss.backward()
        optimizer.step()
        history.append(loss.item())
        
        if (epoch+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{EPOCHS}], RMSE: {loss.item():.4f}')
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        train_pred = model(X_tr).numpy().flatten()
        test_pred = model(X_te).numpy().flatten()
        
    train_rmse = np.sqrt(mean_squared_error(y_tr.numpy(), train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_te.numpy(), test_pred))
    
    return history, train_pred, test_pred, train_rmse, test_rmse

### 6. Run

In [ ]:
# Initialize Models
lstm_model = RULPredictorLSTM(input_dim=len(FEATURES), hidden_dim=64, num_layers=2)
trans_model = RULTransformer(input_dim=len(FEATURES))

# Train & Eval LSTM
lstm_hist, lstm_tr_p, lstm_te_p, lstm_tr_r, lstm_te_r = train_and_evaluate(
    lstm_model, X_train, y_train, X_test, y_test, name="LSTM"
)

# Train & Eval Transformer
trans_hist, trans_tr_p, trans_te_p, trans_tr_r, trans_te_r = train_and_evaluate(
    trans_model, X_train, y_train, X_test, y_test, name="Transformer"
)

# Comparison Summary
print("\n" + "="*30)
print(f"{'Model':<15} | {'Train RMSE':<10} | {'Test RMSE':<10}")
print("-" * 45)
print(f"{'LSTM':<15} | {lstm_tr_r:<10.4f} | {lstm_te_r:<10.4f}")
print(f"{'Transformer':<15} | {trans_tr_r:<10.4f} | {trans_te_r:<10.4f}")
print("="*30)


--- Training LSTM ---
Epoch [10/50], RMSE: 96.2817
Epoch [20/50], RMSE: 95.2663
Epoch [30/50], RMSE: 93.6867
Epoch [40/50], RMSE: 91.9022
Epoch [50/50], RMSE: 89.8588

--- Training Transformer ---


### 7. Visualization 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# A. Training Loss Comparison
axes[0, 0].plot(lstm_hist, label='LSTM')
axes[0, 0].plot(trans_hist, label='Transformer')
axes[0, 0].set_title('Training Convergence (RMSE)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()

# B. LSTM Actual vs Predicted (Test)
axes[0, 1].scatter(y_test.numpy(), lstm_te_p, alpha=0.3, color='blue')
axes[0, 1].plot([0, 125], [0, 125], 'r--')
axes[0, 1].set_title(f'LSTM: Actual vs Predicted (Test RMSE: {lstm_te_r:.2f})')

# C. Transformer Actual vs Predicted (Test)
axes[1, 0].scatter(y_test.numpy(), trans_te_p, alpha=0.3, color='green')
axes[1, 0].plot([0, 125], [0, 125], 'r--')
axes[1, 0].set_title(f'Transformer: Actual vs Predicted (Test RMSE: {trans_te_r:.2f})')

# D. Model Prediction Comparison (First 100 samples of Test)
axes[1, 1].plot(y_test.numpy()[:100], label='Actual', color='black', linewidth=2)
axes[1, 1].plot(lstm_te_p[:100], label='LSTM', linestyle='--')
axes[1, 1].plot(trans_te_p[:100], label='Transformer', linestyle='--')
axes[1, 1].set_title('Sample Prediction Comparison')
axes[1, 1].legend()

plt.tight_layout()
plt.show()